# Member 4 - Logistic Regression
## ML Assignment: FIFA Player Position Prediction

- **Algorithm:** Logistic Regression (Multi-class)
- **Dataset:** FIFA 22 Complete Player Dataset
- **Source:** Kaggle - FIFA 22 Players
- **Task:** Predict player position - ATTACKER, MIDFIELDER, or DEFENDER

This is a standalone version. No external files needed. Data is downloaded automatically.


## 1. Introduction - What is Logistic Regression?

Logistic Regression is a classification algorithm. Even though it has "regression" in the name, it is used to **predict categories**, not numbers.

It works by:
1. Taking the input features and calculating a score (called `z`)
2. Passing that score through a **softmax function** to get probabilities for each class
3. Picking the class with the **highest probability**

**Why use it for FIFA position prediction?**

- It is easy to understand - the model shows which stats matter most
- It gives a confidence score for each position
- It trains very fast, even on thousands of players
- It is a good starting point to compare with other algorithms
- One limitation: it can only draw a straight-line boundary between classes


## 2. Import Librarie


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time
warnings.filterwarnings('ignore')

# Machine learning tools
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, learning_curve
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, f1_score, precision_score, recall_score
)
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# Plot settings
plt.rcParams["figure.dpi"] = 110
sns.set_theme(style="whitegrid", palette="muted")

print("All libraries imported successfully!")

## 3. Load and Prepare the Dataset

The dataset comes from Kaggle and contains stats for over 18,000 real football players from FIFA 22.

**Features used to train the model:**

- `age` - player age
- `height_cm` - height in centimetres
- `weight_kg` - weight in kilograms
- `overall` - skill rating (0–100)
- `potential` - potential rating (0–100)
- `pace` - speed
- `shooting` - shooting accuracy
- `passing` - passing accuracy
- `dribbling` - ball control
- `defending` - defensive skill
- `physic` - physical strength

**Target label:** ATTACKER, MIDFIELDER, or DEFENDER


In [ ]:
# Load the shared data files — same files used by all 4 members
X_train     = np.load("X_train.npy")
X_test      = np.load("X_test.npy")
y_train     = np.load("y_train.npy")
y_test      = np.load("y_test.npy")
class_names = np.load("class_names.npy", allow_pickle=True)

FEATURES = ["age", "height_cm", "weight_kg", "overall", "potential",
            "pace", "shooting", "passing", "dribbling", "defending", "physic"]

print("Shared data loaded!")
print(f"Training samples : {len(X_train):,}")
print(f"Testing  samples : {len(X_test):,}")
print(f"Features         : {X_train.shape[1]}")
print(f"Classes          : {list(class_names)}")
print()

# Show class distribution
unique, counts = np.unique(y_train, return_counts=True)
print("Training class distribution:")
for idx, cnt in zip(unique, counts):
    print(f"  {class_names[idx]:<12}: {cnt:,} samples ({cnt/len(y_train)*100:.1f}%)")

## 4. Train the Logistic Regression Model


In [ ]:
# Create the Logistic Regression model
# solver='lbfgs'  : works well for multi-class problems
# C=1.0           : regularisation strength (higher = less regularisation)
# max_iter=1000   : maximum number of steps to find the best weights
# random_state=42 : makes results reproducible
lr_model = LogisticRegression(
    solver='lbfgs',
    C=1.0,
    max_iter=1000,
    random_state=42
)

print('Training Logistic Regression...')
start = time.time()
lr_model.fit(X_train, y_train)
elapsed = time.time() - start

print(f'Training complete!')
print(f'   Time taken   : {elapsed:.3f} seconds')
print(f'   Iterations   : {lr_model.n_iter_[0]}')
print(f'   Converged    : {lr_model.n_iter_[0] < lr_model.max_iter}')

## 6. Evaluate the Model

### 6.1 Accuracy and Classification Report


In [ ]:
# Make predictions on the test set
y_pred   = lr_model.predict(X_test)
y_proba  = lr_model.predict_proba(X_test)

# Calculate accuracy on both training and test sets
test_acc  = accuracy_score(y_test,  y_pred)
train_acc = accuracy_score(y_train, lr_model.predict(X_train))

# AUC-ROC: measures how well the model separates the three classes
auc_score = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print("=" * 55)
print("    LOGISTIC REGRESSION — EVALUATION RESULTS")
print("=" * 55)
print(f"  Train Accuracy  : {train_acc * 100:.2f}%")
print(f"  Test  Accuracy  : {test_acc  * 100:.2f}%")
print(f"  AUC-ROC (macro) : {auc_score:.4f}")
gap = (train_acc - test_acc) * 100
print(f"  Overfitting Gap : {gap:.2f}%", end="  ")
print("Good generalisation" if gap < 3 else "Consider lower C")
print()
print("Per-Class Report:")
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

### 6.2 Precision, Recall and F1-Score

In [ ]:
precision_macro    = precision_score(y_test, y_pred, average="macro",    zero_division=0)
precision_weighted = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall_macro       = recall_score   (y_test, y_pred, average="macro",    zero_division=0)
recall_weighted    = recall_score   (y_test, y_pred, average="weighted", zero_division=0)
f1_macro           = f1_score       (y_test, y_pred, average="macro",    zero_division=0)
f1_weighted        = f1_score       (y_test, y_pred, average="weighted", zero_division=0)

print("=== PRECISION, RECALL AND F1-SCORE ===")
print(f"  Precision  (macro)    : {precision_macro:.4f}")
print(f"  Precision  (weighted) : {precision_weighted:.4f}")
print(f"  Recall     (macro)    : {recall_macro:.4f}")
print(f"  Recall     (weighted) : {recall_weighted:.4f}")
print(f"  F1-Score   (macro)    : {f1_macro:.4f}")
print(f"  F1-Score   (weighted) : {f1_weighted:.4f}")
print()

# Per-class breakdown
precision_per = precision_score(y_test, y_pred, average=None, zero_division=0)
recall_per    = recall_score   (y_test, y_pred, average=None, zero_division=0)
f1_per        = f1_score       (y_test, y_pred, average=None, zero_division=0)

print(f'{"Class":<20} {"Precision":>10} {"Recall":>10} {"F1-Score":>10}')
print("-" * 52)
for i, cls in enumerate(class_names):
    print(f"{cls:<20} {precision_per[i]:>10.4f} {recall_per[i]:>10.4f} {f1_per[i]:>10.4f}")

### 6.3 Confusion Matrix

In [ ]:
# Build the confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Right: percentage per row (shows recall per class)
sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='YlOrRd',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[1], linewidths=0.5)
axes[1].set_title('Confusion Matrix (Normalised %)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.suptitle('Logistic Regression — Confusion Matrices', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('member4_confusion_matrix.png', bbox_inches='tight')
plt.show()

print('\nPer-Class Recall:')
for i, cls in enumerate(class_names):
    recall = cm[i,i] / cm[i].sum() * 100
    print(f'  {cls:<12}: {cm[i,i]}/{cm[i].sum()} correct  ({recall:.1f}% recall)')